# Выявление автоматизированного трафика

**Цель.** Для каждой куки из теста получить `score` в диапазоне [0, 1].
Класс `1` — трафик известных сервисов сбора данных. Единица оценки — кука.

**Основная метрика.** Максимальный precision при recall не ниже 70%.
Выбор решения основан на объединённых временных OOF-прогнозах фолдов 0–2.
Поздний фолд 3 участвовал в выборе метода и показан отдельно.

**Правило времени.** В признаках используются события внутри суточного окна
куки. История других кук доступна после завершения их окон. Для
сглаживания обучающих прогнозов берутся только OOF-score.

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits

root = Path(os.environ.get("AVITO_PROJECT_DIR", Path.cwd())).resolve()
output = root / "result"
output.mkdir(exist_ok=True)
print("Python", sys.version.split()[0])


Python 3.12.14


## Данные и качество наблюдений

Одна строка метаданных задаёт куку и границы её окна. События фильтруются
по полуоткрытому интервалу `[window_start_ts, window_end_ts)`; полные
дубликаты удаляются до расчёта признаков.

In [2]:
"""Загрузка данных и отбор событий в окне [начало, конец)."""

from __future__ import annotations

from pathlib import Path

import pandas as pd

TIME_COLUMNS = ["cookie_created_at", "window_start_ts", "window_end_ts"]


def load_data(data_dir):
    data_dir = Path(data_dir)
    train = pd.read_csv(data_dir / "train.csv")
    test = pd.read_csv(data_dir / "test.csv")
    events = pd.read_csv(data_dir / "events.csv.gz", compression="gzip")
    for frame in (train, test):
        for column in TIME_COLUMNS:
            frame[column] = pd.to_datetime(frame[column], errors="raise")
        assert frame.cookie_id.notna().all() and frame.cookie_id.is_unique
        assert (frame.window_start_ts < frame.window_end_ts).all()
    events["event_ts"] = pd.to_datetime(events.event_ts, errors="raise")
    assert "target" not in test.columns and "target" not in events.columns
    assert set(train.target.unique()) <= {0, 1}
    assert not set(train.cookie_id) & set(test.cookie_id)
    return train, test, events


def window_events(events, metadata, *, deduplicate=True):
    """Отобрать события до расчёта признаков. eid обозначает тип события.

    При включённой очистке удаляются только полностью одинаковые строки.
    Разные события с одинаковым временем сохраняются: их порядок неизвестен,
    поэтому признаки последовательности отдельно учитывают такие совпадения.
    """
    original_columns = list(events.columns)
    joined = events.merge(
        metadata[["cookie_id", "window_start_ts", "window_end_ts"]],
        on="cookie_id",
        how="inner",
        validate="many_to_one",
    )
    mask = (joined.event_ts >= joined.window_start_ts) & (
        joined.event_ts < joined.window_end_ts
    )
    out = joined.loc[mask, original_columns].copy()
    if deduplicate:
        out = out.drop_duplicates(subset=original_columns)
    assert out.event_ts.notna().all()
    return out.reset_index(drop=True)


In [3]:
train, test, events = load_data(root / "data")
metadata = pd.concat([train.drop(columns="target"), test], ignore_index=True)
windowed = window_events(events, metadata)
summary = pd.DataFrame(
    {
        "sample": ["train", "test"],
        "cookies": [len(train), len(test)],
        "first_day": [
            train.window_start_ts.min().date(),
            test.window_start_ts.min().date(),
        ],
        "last_day": [
            train.window_start_ts.max().date(),
            test.window_start_ts.max().date(),
        ],
    }
)
print(summary.to_string(index=False))
print("events:", len(events), "in window:", len(windowed))
print("positive rate:", f"{train.target.mean():.2%}")


sample  cookies  first_day   last_day
 train    11091 2026-04-06 2026-04-19
  test     4909 2026-04-20 2026-04-26
events: 328905 in window: 283833
positive rate: 8.11%


## Признаки

Поведенческие признаки описывают объём действий, интервалы, просмотры,
поиск, User-Agent и курсор. Словарь переходов получен на раннем обучающем
периоде. Исторические признаки используют только окна, завершившиеся
не позже начала текущего окна. Метка `target` в признаки не входит.

In [4]:
"""Признаки куки из событий без использования целевых меток.

Признаки объединены в группы. Частоты категорий считаются отдельно внутри
каждого обучающего фолда. Словарь переходов берётся из раннего периода.
"""

from __future__ import annotations

import re

import numpy as np
import pandas as pd


def normalized_platform(series):
    return (
        series.fillna("unknown")
        .str.strip()
        .str.lower()
        .replace({"desktop": "web", "iphone": "ios"})
    )


def parse_ua(text):
    """Разобрать известные форматы User-Agent; незнакомые отнести к other."""
    text = str(text)
    low = text.lower()
    http = any(
        k in low
        for k in [
            "python-requests",
            "python-urllib",
            "go-http-client",
            "curl/",
            "node-fetch",
            "scrapy/",
        ]
    )
    if low.startswith("avito/"):
        family, pattern = "app", r"Avito/(\d+)"
    elif "headlesschrome" in low:
        family, pattern = "headless", r"HeadlessChrome/(\d+)"
    elif "yabrowser" in low:
        family, pattern = "yandex", r"YaBrowser/(\d+)"
    elif "firefox" in low:
        family, pattern = "firefox", r"Firefox/(\d+)"
    elif "chrome" in low:
        family, pattern = "chrome", r"Chrome/(\d+)"
    elif "safari" in low:
        family, pattern = "safari", r"Version/(\d+)"
    elif http:
        family, pattern = "http_client", r"/(\d+)"
    else:
        family, pattern = "other", r"(?!)"
    if "android" in low:
        os_family = "android"
    elif any(x in low for x in ["iphone", "ipad", "ios "]):
        os_family = "ios"
    elif "windows" in low:
        os_family = "windows"
    elif "macintosh" in low:
        os_family = "mac"
    elif "linux" in low:
        os_family = "linux"
    else:
        os_family = "other"
    match = re.search(pattern, text, re.IGNORECASE)
    return {
        "family": family,
        "os": os_family,
        "major": float(match.group(1)) if match else np.nan,
        "mobile": os_family in ["android", "ios"],
        "headless": "headless" in low,
        "http_client": http,
        "bot_keyword": any(
            x in low for x in ["bot", "crawler", "spider", "scrapy"]
        ),
        "tablet": "ipad" in low or "tablet" in low,
    }


def fit_config(events, warmup_metadata):
    ev = window_events(events, warmup_metadata, deduplicate=True)
    unique_time = ~ev.duplicated(["cookie_id", "event_ts"], keep=False)
    ordered = ev.loc[unique_time].sort_values(["cookie_id", "event_ts"])
    previous = ordered.groupby("cookie_id").event_name.shift()
    pairs = (previous + ">" + ordered.event_name).dropna()
    common = pairs.value_counts()
    return {
        "version": 1,
        "event_names": sorted(ev.event_name.unique().tolist()),
        "bigrams": common.loc[common >= 30].head(12).index.tolist(),
        "vocabulary_train_end_exclusive": str(
            warmup_metadata.window_end_ts.max()
        ),
        "deduplicate": True,
        "session_gaps_seconds": [30, 60, 300, 1800],
    }


def build_features(events, metadata, config):
    ev = window_events(
        events, metadata, deduplicate=config.get("deduplicate", True)
    )
    ev = ev.sort_values(
        ["cookie_id", "event_ts"], kind="mergesort"
    ).reset_index(drop=True)
    index = pd.Index(metadata.cookie_id, name="cookie_id")
    meta = metadata.set_index("cookie_id")
    features = {}

    def add(group, name, value):
        features[group + "__" + name] = (
            value.reindex(index)
            if isinstance(value, pd.Series)
            else pd.Series(value, index=index)
        )

    def stats(
        group,
        prefix,
        values,
        quantiles=(0.1, 0.25, 0.5, 0.75, 0.9),
        all_stats=True,
    ):
        data = pd.DataFrame(
            {"cookie_id": ev.cookie_id, "value": values}
        ).dropna(subset=["value"])
        grouped = data.groupby("cookie_id").value
        for name, series in [
            ("mean", grouped.mean()),
            ("median", grouped.median()),
            ("std", grouped.std(ddof=0)),
            ("min", grouped.min()),
            ("max", grouped.max()),
        ]:
            add(group, prefix + "_" + name, series)
        for q in quantiles:
            add(group, prefix + f"_q{int(q * 100):02d}", grouped.quantile(q))

    def concentration(frame, column, group, prefix):
        sub = frame[["cookie_id", column]].dropna()
        counts = (
            sub.groupby(["cookie_id", column], sort=False)
            .size()
            .rename("count")
        )
        total = counts.groupby(level=0).sum()
        unique = counts.groupby(level=0).size()
        p = counts / counts.groupby(level=0).transform("sum")
        ent = -(p * np.log(p)).groupby(level=0).sum()
        add(group, prefix + "_entropy", ent)
        add(
            group,
            prefix + "_normalized_entropy",
            ent / np.log(unique).replace(0, np.nan),
        )
        add(group, prefix + "_hhi", (p * p).groupby(level=0).sum())
        add(group, prefix + "_top1_share", p.groupby(level=0).max())
        flat = counts.reset_index().sort_values("count", ascending=False)
        top3 = (
            flat.groupby("cookie_id", sort=False)
            .head(3)
            .groupby("cookie_id")["count"]
            .sum()
            / total
        )
        add(group, prefix + "_top3_share", top3)
        # Для одного значения нормированная энтропия равна нулю.
        key = group + "__" + prefix + "_normalized_entropy"
        features[key] = features[key].mask(unique.reindex(index).eq(1), 0)
        return total, unique

    # Размер, возраст и положение окна во времени.
    g = ev.groupby("cookie_id", sort=False)
    n = g.size()
    add("basic", "n_events", n)
    add("basic", "log_n_events", np.log1p(n))
    add("basic", "n_event_types", g.event_name.nunique())
    add("basic", "n_eid_types", g.eid.nunique())
    age = (
        meta.window_start_ts - meta.cookie_created_at
    ).dt.total_seconds() / 86400
    add("metadata", "age_days", age)
    add("metadata", "log_age_days", np.log1p(age))
    window = (meta.window_end_ts - meta.window_start_ts).dt.total_seconds()
    add("metadata", "window_hours", window / 3600)
    for part, series in [
        ("start_hour", meta.window_start_ts.dt.hour),
        ("weekday", meta.window_start_ts.dt.dayofweek),
        ("weekend", meta.window_start_ts.dt.dayofweek.ge(5).astype(int)),
        (
            "start_day",
            (
                meta.window_start_ts - pd.Timestamp("1970-01-01")
            ).dt.total_seconds()
            / 86400,
        ),
        (
            "created_day",
            (
                meta.cookie_created_at - pd.Timestamp("1970-01-01")
            ).dt.total_seconds()
            / 86400,
        ),
    ]:
        add("calendar", part, series)
    # Частоты действий и разнообразие типов событий.
    event_counts = pd.crosstab(ev.cookie_id, ev.event_name)
    for name in config["event_names"]:
        count = (
            event_counts[name]
            if name in event_counts
            else pd.Series(0, index=n.index)
        )
        add("event", name + "_count", count)
        add("event", name + "_share", count / n)
        add("event", name + "_present", count.gt(0).astype(int))
        add("event", name + "_log_count", np.log1p(count))
    add(
        "event",
        "unknown_count",
        ev.assign(unknown=~ev.event_name.isin(config["event_names"]))
        .groupby("cookie_id")
        .unknown.sum(),
    )
    concentration(ev, "event_name", "entropy", "event")

    # Интенсивность, интервалы между событиями и длина сессий.
    start = ev.cookie_id.map(meta.window_start_ts)
    ev["_offset"] = (ev.event_ts - start).dt.total_seconds()
    first = g.event_ts.min()
    last = g.event_ts.max()
    duration = (last - first).dt.total_seconds()
    add("temporal", "active_duration", duration)
    add(
        "temporal",
        "first_offset",
        (first - meta.window_start_ts).dt.total_seconds(),
    )
    add(
        "temporal",
        "last_offset",
        (last - meta.window_start_ts).dt.total_seconds(),
    )
    add("temporal", "tail_gap", (meta.window_end_ts - last).dt.total_seconds())
    add(
        "temporal", "events_per_active_second", n / duration.replace(0, np.nan)
    )
    add("temporal", "events_per_window_hour", n / (window / 3600))
    delta = g.event_ts.diff().dt.total_seconds()
    stats(
        "temporal",
        "delta",
        delta,
        (0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99),
    )
    dg = delta.groupby(ev.cookie_id)
    mean = dg.mean()
    std = dg.std(ddof=0)
    add("temporal", "delta_cv", std / mean.replace(0, np.nan))
    add(
        "temporal",
        "burstiness",
        (std - mean) / (std + mean).replace(0, np.nan),
    )
    add("temporal", "n_unique_delta", dg.nunique())
    valid = delta.notna()
    for limit in [1, 2, 5, 10, 30]:
        add(
            "temporal",
            f"delta_lt_{limit}_share",
            (delta.lt(limit) & valid).groupby(ev.cookie_id).sum()
            / valid.groupby(ev.cookie_id).sum().replace(0, np.nan),
        )
    dc = (
        pd.DataFrame({"cookie_id": ev.cookie_id, "delta": delta})
        .dropna()
        .groupby(["cookie_id", "delta"])
        .size()
    )
    add(
        "temporal",
        "delta_mode_share",
        dc.groupby(level=0).max() / dc.groupby(level=0).sum(),
    )
    add(
        "temporal",
        "delta_repeat_share",
        1 - dg.nunique() / dg.count().replace(0, np.nan),
    )
    for seconds in [1, 60, 300, 900, 3600]:
        bucket = (ev._offset // seconds).astype(int)
        bc = (
            ev.assign(bucket=bucket)
            .groupby(["cookie_id", "bucket"])
            .size()
            .rename("count")
        )
        bg = bc.groupby(level=0)
        prefix = f"bucket_{seconds}s"
        for name, series in [
            ("active", bg.size()),
            ("mean", bg.mean()),
            ("std", bg.std(ddof=0)),
            ("max", bg.max()),
            ("median", bg.median()),
            ("q95", bg.quantile(0.95)),
        ]:
            add("activity", prefix + "_" + name, series)
        prob = bc / bc.groupby(level=0).transform("sum")
        add(
            "entropy",
            prefix + "_entropy",
            -(prob * np.log(prob)).groupby(level=0).sum(),
        )
        add("activity", prefix + "_top1_share", bg.max() / n)
    for gap in config["session_gaps_seconds"]:
        sessions = (
            (delta.isna() | delta.gt(gap)).groupby(ev.cookie_id).cumsum()
        )
        sg = (
            ev.assign(session=sessions)
            .groupby(["cookie_id", "session"])
            .event_ts.agg(["min", "max", "size"])
        )
        durations = (sg["max"] - sg["min"]).dt.total_seconds()
        prefix = f"session_{gap}s"
        for name, series in [
            ("count", sg.groupby(level=0).size()),
            ("mean_duration", durations.groupby(level=0).mean()),
            ("max_duration", durations.groupby(level=0).max()),
            ("duration_std", durations.groupby(level=0).std(ddof=0)),
            ("mean_events", sg["size"].groupby(level=0).mean()),
            ("max_events", sg["size"].groupby(level=0).max()),
        ]:
            add("activity", prefix + "_" + name, series)

    # Переходы считаем для событий с уникальным временем.
    # Порядок событий внутри секунды неизвестен.
    tied = ev.duplicated(["cookie_id", "event_ts"], keep=False)
    add("sequence", "ambiguous_time_share", tied.groupby(ev.cookie_id).mean())
    sq = ev.loc[~tied, ["cookie_id", "event_ts", "event_name"]].copy()
    prev = sq.groupby("cookie_id").event_name.shift()
    sq["transition"] = prev + ">" + sq.event_name
    trans = sq.dropna(subset=["transition"])
    nt = trans.groupby("cookie_id").size()
    nu = trans.groupby("cookie_id").transition.nunique()
    add("sequence", "n_transitions", nt)
    add("sequence", "n_unique_transitions", nu)
    add("sequence", "repeat_transition_share", 1 - nu / nt)
    add(
        "sequence",
        "self_transition_share",
        (sq.event_name.eq(prev) & prev.notna()).groupby(sq.cookie_id).sum()
        / nt,
    )
    concentration(trans, "transition", "sequence", "transition")
    runs = sq.event_name.ne(prev).groupby(sq.cookie_id).cumsum()
    rc = sq.assign(run=runs).groupby(["cookie_id", "run"]).size()
    add("sequence", "longest_run", rc.groupby(level=0).max())
    add("sequence", "mean_run", rc.groupby(level=0).mean())
    add("sequence", "n_runs", rc.groupby(level=0).size())
    tc = pd.crosstab(trans.cookie_id, trans.transition)
    for i, pair in enumerate(config["bigrams"]):
        val = tc[pair] if pair in tc else pd.Series(0, index=n.index)
        add("sequence", f"bigram_{i}_share", val / nt)

    # Просмотренные объявления, категории и повторы просмотров.
    item = ev.loc[ev.item_id.notna()].copy()
    ni = item.groupby("cookie_id").size()
    ui = item.groupby("cookie_id").item_id.nunique()
    add("item", "n_item_events", ni)
    add("item", "n_unique_items", ui)
    add("item", "events_per_item", ni / ui)
    add("item", "repeat_item_share", 1 - ui / ni)
    add("item", "revisit_count", ni - ui)
    for col, prefix in [
        ("item_id", "item"),
        ("item_category", "category"),
        ("item_location", "location"),
        ("seller_type", "seller"),
    ]:
        add(
            "item",
            prefix + "_n_unique",
            item.groupby("cookie_id")[col].nunique(),
        )
        add(
            "item",
            prefix + "_missing_share",
            item[col].isna().groupby(item.cookie_id).mean(),
        )
        concentration(item, col, "item", prefix)
    add(
        "item",
        "seller_pro_share",
        item.seller_type.eq("pro").groupby(item.cookie_id).sum()
        / item.seller_type.notna().groupby(item.cookie_id).sum(),
    )
    it = item.loc[~item.duplicated(["cookie_id", "event_ts"], keep=False)]
    for c, prefix in [
        ("item_id", "item"),
        ("item_category", "category"),
        ("item_location", "location"),
    ]:
        previous = it.groupby("cookie_id")[c].shift()
        eligible = previous.notna() & it[c].notna()
        add(
            "item",
            "consecutive_same_" + prefix,
            (it[c].eq(previous) & eligible).groupby(it.cookie_id).sum()
            / eligible.groupby(it.cookie_id).sum().replace(0, np.nan),
        )

    # Поисковые запросы и страницы выдачи.
    search = ev.loc[ev.event_name.eq("search_results_view")].copy()
    ns = search.groupby("cookie_id").size()
    uq = search.groupby("cookie_id").search_query.nunique()
    add("search", "n_search_events", ns)
    add("search", "n_unique_queries", uq)
    add("search", "repeat_query_share", 1 - uq / ns)
    concentration(search, "search_query", "search", "query")
    chars = ev.search_query.str.len()
    tokens = ev.search_query.str.split().str.len()
    stats("search", "query_chars", chars, (0.9,))
    stats("search", "query_tokens", tokens, ())
    for name, regex in [
        ("digit", r"\d"),
        ("alpha", r"[^\W\d_]"),
        ("space", r"\s"),
        ("cyrillic", r"[А-Яа-яЁё]"),
        ("latin", r"[A-Za-z]"),
    ]:
        proportion = ev.search_query.str.count(regex) / chars.replace(
            0, np.nan
        )
        add("search", name + "_share", proportion.groupby(ev.cookie_id).mean())
    stats("search", "page", ev.search_page, (0.9,))
    add(
        "search",
        "n_unique_pages",
        search.groupby("cookie_id").search_page.nunique(),
    )
    ss = search.loc[~search.duplicated(["cookie_id", "event_ts"], keep=False)]
    prevpage = ss.groupby("cookie_id").search_page.shift()
    prevquery = ss.groupby("cookie_id").search_query.shift()
    eligible = ss.search_query.eq(prevquery) & prevpage.notna()
    diff = ss.search_page - prevpage
    for name, flag in [
        ("page_increment", diff.eq(1)),
        ("page_nondecreasing", diff.ge(0)),
        ("page_reset", diff.lt(0)),
        ("page_repeat", diff.eq(0)),
    ]:
        add(
            "search",
            name + "_share",
            (flag & eligible).groupby(ss.cookie_id).sum()
            / eligible.groupby(ss.cookie_id).sum().replace(0, np.nan),
        )
    add(
        "search",
        "deep_page_share",
        ev.search_page.gt(5).groupby(ev.cookie_id).sum() / ns,
    )

    # Платформа и признаки из User-Agent.
    ev["_platform"] = normalized_platform(ev.platform)
    parsed = {ua: parse_ua(ua) for ua in ev.user_agent.dropna().unique()}
    for key in [
        "family",
        "os",
        "major",
        "mobile",
        "headless",
        "http_client",
        "bot_keyword",
        "tablet",
    ]:
        ev["_ua_" + key] = ev.user_agent.map(
            {ua: d[key] for ua, d in parsed.items()}
        )
    for family in [
        "app",
        "headless",
        "yandex",
        "firefox",
        "chrome",
        "safari",
        "http_client",
        "other",
    ]:
        add(
            "ua",
            "family_" + family + "_share",
            ev._ua_family.eq(family).groupby(ev.cookie_id).mean(),
        )
    for os_name in ["android", "ios", "windows", "mac", "linux", "other"]:
        add(
            "ua",
            "os_" + os_name + "_share",
            ev._ua_os.eq(os_name).groupby(ev.cookie_id).mean(),
        )
    for key in ["mobile", "headless", "http_client", "bot_keyword", "tablet"]:
        add(
            "ua",
            key + "_share",
            ev["_ua_" + key].astype(float).groupby(ev.cookie_id).mean(),
        )
    for platform in ["web", "android", "ios", "unknown"]:
        add(
            "ua",
            "platform_" + platform + "_share",
            ev._platform.eq(platform).groupby(ev.cookie_id).mean(),
        )
    add("ua", "n_unique_raw_ua", g.user_agent.nunique())
    add("ua", "n_unique_platform", ev.groupby("cookie_id")._platform.nunique())
    concentration(ev, "user_agent", "ua", "ua")
    uprev = ev.groupby("cookie_id").user_agent.shift()
    add(
        "ua",
        "ua_change_share",
        (uprev.notna() & ev.user_agent.ne(uprev)).groupby(ev.cookie_id).sum()
        / (n - 1).replace(0, np.nan),
    )
    inconsistent = (ev._platform.eq("android") & ev._ua_os.ne("android")) | (
        ev._platform.eq("ios") & ev._ua_os.ne("ios")
    )
    add(
        "ua",
        "platform_os_mismatch_share",
        inconsistent.groupby(ev.cookie_id).mean(),
    )
    stats("ua_version", "browser_major", ev._ua_major, ())

    # Координаты курсора, если платформа их передала.
    ptr = ev.loc[ev.pointer_x.notna() & ev.pointer_y.notna()].copy()
    pn = ptr.groupby("cookie_id").size()
    add("pointer", "n_points", pn)
    add("pointer", "coverage_all", pn / n)
    web = ev._platform.eq("web").groupby(ev.cookie_id).sum()
    add("pointer", "coverage_web", pn / web.replace(0, np.nan))
    for axis in ["x", "y"]:
        col = "pointer_" + axis
        stats("pointer", axis, ev[col], (0.1, 0.25, 0.75, 0.9))
        add(
            "pointer",
            axis + "_nunique",
            ptr.groupby("cookie_id")[col].nunique(),
        )
    pairs = ptr.groupby(["cookie_id", "pointer_x", "pointer_y"]).size()
    add("pointer", "n_unique_pairs", pairs.groupby(level=0).size())
    add("pointer", "repeat_pair_share", 1 - pairs.groupby(level=0).size() / pn)
    add("pointer", "dominant_pair_share", pairs.groupby(level=0).max() / pn)
    pg = ptr.groupby("cookie_id")
    width = pg.pointer_x.max() - pg.pointer_x.min()
    height = pg.pointer_y.max() - pg.pointer_y.min()
    add("pointer", "bbox_width", width)
    add("pointer", "bbox_height", height)
    add("pointer", "bbox_area", width * height)
    # Разность координат соседних событий описывает смещение курсора.
    pp = ptr.loc[~ptr.duplicated(["cookie_id", "event_ts"], keep=False)].copy()
    dx = pp.groupby("cookie_id").pointer_x.diff()
    dy = pp.groupby("cookie_id").pointer_y.diff()
    distance = np.hypot(dx, dy)
    dgroup = distance.groupby(pp.cookie_id)
    for name, series in [
        ("distance_mean", dgroup.mean()),
        ("distance_median", dgroup.median()),
        ("distance_std", dgroup.std(ddof=0)),
        ("distance_max", dgroup.max()),
        ("path_sum", dgroup.sum(min_count=1)),
        ("abs_dx_mean", dx.abs().groupby(pp.cookie_id).mean()),
        ("abs_dy_mean", dy.abs().groupby(pp.cookie_id).mean()),
    ]:
        add("pointer", name, series)
    valid = distance.notna()
    for name, flag in [
        ("zero_movement", distance.eq(0)),
        ("same_x", dx.eq(0)),
        ("same_y", dy.eq(0)),
    ]:
        add(
            "pointer",
            name + "_share",
            (flag & valid).groupby(pp.cookie_id).sum()
            / valid.groupby(pp.cookie_id).sum().replace(0, np.nan),
        )
    ppg = pp.groupby("cookie_id")
    displacement = np.hypot(
        ppg.pointer_x.last() - ppg.pointer_x.first(),
        ppg.pointer_y.last() - ppg.pointer_y.first(),
    )
    add(
        "pointer",
        "path_efficiency",
        displacement / dgroup.sum().replace(0, np.nan),
    )

    def count(name):
        return (
            event_counts[name].reindex(index).fillna(0)
            if name in event_counts
            else pd.Series(0, index=index)
        )

    # Соотношения действий и наиболее частые категории.
    views = count("item_view")
    searches = count("search_results_view")
    contacts = (
        count("contact_phone_show")
        + count("contact_chat_open")
        + count("contact_message_sent")
    )
    for name, numerator, denominator in [
        ("unique_items_per_event", ui, n),
        ("views_per_search", views, searches),
        ("contacts_per_view", contacts, views),
        ("favorites_per_view", count("favorite_add"), views),
        ("photos_per_view", count("photo_swipe"), views),
        ("seller_views_per_view", count("seller_page_view"), views),
        ("queries_per_active_minute", uq, duration / 60),
    ]:
        add("ratios", name, numerator / denominator.replace(0, np.nan))
    for frame, column, name in [
        (item, "item_category", "category"),
        (item, "item_location", "location"),
        (search, "search_query", "query"),
        (ev, "_ua_family", "ua_family"),
    ]:
        cnt = (
            frame.groupby(["cookie_id", column], dropna=True)
            .size()
            .reset_index(name="n")
        )
        dominant = (
            cnt.sort_values(
                ["cookie_id", "n", column], ascending=[True, False, True]
            )
            .drop_duplicates("cookie_id")
            .set_index("cookie_id")[column]
        )
        add("rarity", "dominant_" + name, dominant.fillna("__NONE__"))
    result = pd.DataFrame(features, index=index)
    numeric = [c for c in result if not c.startswith("rarity__")]
    result[numeric] = (
        result[numeric].replace([np.inf, -np.inf], np.nan).astype("float32")
    )
    # Отсутствие события даёт нулевой счётчик.
    # Отсутствующее измерение оставляем NaN.
    zero_groups = ["basic__", "event__"]
    for c in result:
        if any(c.startswith(prefix) for prefix in zero_groups):
            result[c] = result[c].fillna(0)
        elif c.startswith("rarity__"):
            result[c] = result[c].fillna("__NONE__").astype(str)
    assert result.index.is_unique and result.index.equals(index)
    assert not any(
        c in result
        for c in ["target", "cookie_id", "eid", "event_ts", "user_agent"]
    )
    return result


In [5]:
"""Признаки общих просмотренных объявлений по завершённым окнам без меток.

Для куки с началом окна в момент T учитываются только другие куки,
чьи окна завершились не позже T.
"""

from __future__ import annotations

from collections import Counter, defaultdict

import numpy as np
import pandas as pd


FEATURE_NAMES = [
    "prior_pool_size",
    "seen_before_share",
    "prior_count_mean",
    "prior_count_max",
    "prior_frequency_mean",
    "seen_by_3_share",
    "recent_7d_count_mean",
    "recent_7d_seen_share",
    "max_cookie_overlap",
    "max_cookie_overlap_share",
    "max_cookie_jaccard",
    "prior_cookie_overlap_3_count",
]


def build_history_features(
    events: pd.DataFrame, metadata: pd.DataFrame
) -> pd.DataFrame:
    ev = window_events(events, metadata, deduplicate=True)
    viewed = ev.loc[
        ev.event_name.eq("item_view") & ev.item_id.notna(),
        ["cookie_id", "item_id"],
    ]
    item_sets = (
        viewed.groupby("cookie_id").item_id.agg(lambda x: set(x)).to_dict()
    )
    by_start = metadata.sort_values(
        ["window_start_ts", "cookie_id"]
    ).reset_index(drop=True)
    by_end = metadata.sort_values(["window_end_ts", "cookie_id"]).reset_index(
        drop=True
    )
    cookie_items: dict[str, set] = {
        cookie: item_sets.get(cookie, set()) for cookie in metadata.cookie_id
    }
    history: dict[object, list[tuple[str, pd.Timestamp]]] = defaultdict(list)
    previous_size: dict[str, int] = {}
    rows: dict[str, dict[str, float]] = {}
    next_completed = 0
    for start, group in by_start.groupby("window_start_ts", sort=True):
        while (
            next_completed < len(by_end)
            and by_end.iloc[next_completed].window_end_ts <= start
        ):
            old = by_end.iloc[next_completed]
            items = cookie_items[old.cookie_id]
            previous_size[old.cookie_id] = len(items)
            for item in items:
                history[item].append((old.cookie_id, old.window_end_ts))
            next_completed += 1
        pool_size = len(previous_size)
        for cookie in group.cookie_id:
            items = cookie_items[cookie]
            counts, recent_counts = [], []
            overlap: Counter[str] = Counter()
            for item in items:
                past = history.get(item, [])
                counts.append(len(past))
                recent_counts.append(
                    sum(
                        (start - ts).total_seconds() <= 7 * 86400
                        for _, ts in past
                    )
                )
                overlap.update(prior_cookie for prior_cookie, _ in past)
            n = len(items)
            best_overlap = max(overlap.values(), default=0)
            best_jaccard = max(
                (
                    shared / (n + previous_size[other] - shared)
                    for other, shared in overlap.items()
                ),
                default=0.0,
            )
            rows[cookie] = {
                "prior_pool_size": float(pool_size),
                "seen_before_share": sum(c > 0 for c in counts) / n
                if n
                else np.nan,
                "prior_count_mean": float(np.mean(counts)) if n else np.nan,
                "prior_count_max": float(max(counts)) if n else np.nan,
                "prior_frequency_mean": float(np.mean(counts)) / pool_size
                if n and pool_size
                else np.nan,
                "seen_by_3_share": sum(c >= 3 for c in counts) / n
                if n
                else np.nan,
                "recent_7d_count_mean": float(np.mean(recent_counts))
                if n
                else np.nan,
                "recent_7d_seen_share": sum(c > 0 for c in recent_counts) / n
                if n
                else np.nan,
                "max_cookie_overlap": float(best_overlap),
                "max_cookie_overlap_share": best_overlap / n if n else np.nan,
                "max_cookie_jaccard": float(best_jaccard),
                "prior_cookie_overlap_3_count": float(
                    sum(c >= 3 for c in overlap.values())
                ),
            }
    result = pd.DataFrame.from_dict(rows, orient="index").reindex(
        metadata.cookie_id
    )
    result.index = pd.Index(metadata.cookie_id, name="cookie_id")
    assert result.index.is_unique and list(result.columns) == FEATURE_NAMES
    return result.add_prefix("item_history__").astype("float32")


In [6]:
# Наборы признаков зафиксированы на ранней обучающей части.
MODEL_SEED = 20260926

BASE_FEATURES = """
basic__n_events basic__log_n_events basic__n_event_types basic__n_eid_types
metadata__age_days metadata__log_age_days metadata__window_hours
event__contact_chat_open_count event__contact_chat_open_share
event__contact_chat_open_present event__contact_chat_open_log_count
event__contact_message_sent_count event__contact_message_sent_share
event__contact_message_sent_present event__contact_message_sent_log_count
event__contact_phone_show_count event__contact_phone_show_share
event__contact_phone_show_present event__contact_phone_show_log_count
event__favorite_add_count event__favorite_add_share
event__favorite_add_present event__favorite_add_log_count
event__item_view_count event__item_view_share event__item_view_present
event__item_view_log_count event__login_count event__login_share
event__login_present event__login_log_count event__photo_swipe_count
event__photo_swipe_share event__photo_swipe_present
event__photo_swipe_log_count event__search_results_view_count
event__search_results_view_share event__search_results_view_present
event__search_results_view_log_count event__seller_page_view_count
event__seller_page_view_share event__seller_page_view_present
event__seller_page_view_log_count event__unknown_count entropy__event_entropy
entropy__event_normalized_entropy entropy__event_hhi entropy__event_top1_share
entropy__event_top3_share temporal__active_duration temporal__first_offset
temporal__last_offset temporal__tail_gap temporal__events_per_active_second
temporal__events_per_window_hour temporal__delta_mean temporal__delta_median
temporal__delta_std temporal__delta_min temporal__delta_max
temporal__delta_q01 temporal__delta_q05 temporal__delta_q10
temporal__delta_q25 temporal__delta_q50 temporal__delta_q75
temporal__delta_q90 temporal__delta_q95 temporal__delta_q99 temporal__delta_cv
temporal__burstiness temporal__n_unique_delta temporal__delta_lt_1_share
temporal__delta_lt_2_share temporal__delta_lt_5_share
temporal__delta_lt_10_share temporal__delta_lt_30_share
temporal__delta_mode_share temporal__delta_repeat_share
activity__bucket_1s_active activity__bucket_1s_mean activity__bucket_1s_std
activity__bucket_1s_max activity__bucket_1s_median activity__bucket_1s_q95
entropy__bucket_1s_entropy activity__bucket_1s_top1_share
activity__bucket_60s_active activity__bucket_60s_mean activity__bucket_60s_std
activity__bucket_60s_max activity__bucket_60s_median activity__bucket_60s_q95
entropy__bucket_60s_entropy activity__bucket_60s_top1_share
activity__bucket_300s_active activity__bucket_300s_mean
activity__bucket_300s_std activity__bucket_300s_max
activity__bucket_300s_median activity__bucket_300s_q95
entropy__bucket_300s_entropy activity__bucket_300s_top1_share
activity__bucket_900s_active activity__bucket_900s_mean
activity__bucket_900s_std activity__bucket_900s_max
activity__bucket_900s_median activity__bucket_900s_q95
entropy__bucket_900s_entropy activity__bucket_900s_top1_share
activity__bucket_3600s_active activity__bucket_3600s_mean
activity__bucket_3600s_std activity__bucket_3600s_max
activity__bucket_3600s_median activity__bucket_3600s_q95
entropy__bucket_3600s_entropy activity__bucket_3600s_top1_share
activity__session_30s_count activity__session_30s_mean_duration
activity__session_30s_max_duration activity__session_30s_duration_std
activity__session_30s_mean_events activity__session_30s_max_events
activity__session_60s_count activity__session_60s_mean_duration
activity__session_60s_max_duration activity__session_60s_duration_std
activity__session_60s_mean_events activity__session_60s_max_events
activity__session_300s_count activity__session_300s_mean_duration
activity__session_300s_max_duration activity__session_300s_duration_std
activity__session_300s_mean_events activity__session_300s_max_events
activity__session_1800s_count activity__session_1800s_mean_duration
activity__session_1800s_max_duration activity__session_1800s_duration_std
activity__session_1800s_mean_events activity__session_1800s_max_events
sequence__ambiguous_time_share sequence__n_transitions
sequence__n_unique_transitions sequence__repeat_transition_share
sequence__self_transition_share sequence__transition_entropy
sequence__transition_normalized_entropy sequence__transition_hhi
sequence__transition_top1_share sequence__transition_top3_share
sequence__longest_run sequence__mean_run sequence__n_runs
sequence__bigram_0_share sequence__bigram_1_share sequence__bigram_2_share
sequence__bigram_3_share sequence__bigram_4_share sequence__bigram_5_share
sequence__bigram_6_share sequence__bigram_7_share sequence__bigram_8_share
sequence__bigram_9_share sequence__bigram_10_share sequence__bigram_11_share
item__n_item_events item__n_unique_items item__events_per_item
item__repeat_item_share item__revisit_count item__item_n_unique
item__item_missing_share item__item_entropy item__item_normalized_entropy
item__item_hhi item__item_top1_share item__item_top3_share
item__category_n_unique item__category_missing_share item__category_entropy
item__category_normalized_entropy item__category_hhi item__category_top1_share
item__category_top3_share item__location_n_unique item__location_missing_share
item__location_entropy item__location_normalized_entropy item__location_hhi
item__location_top1_share item__location_top3_share item__seller_n_unique
item__seller_missing_share item__seller_entropy
item__seller_normalized_entropy item__seller_hhi item__seller_top1_share
item__seller_top3_share item__seller_pro_share item__consecutive_same_item
item__consecutive_same_category item__consecutive_same_location
search__n_search_events search__n_unique_queries search__repeat_query_share
search__query_entropy search__query_normalized_entropy search__query_hhi
search__query_top1_share search__query_top3_share search__query_chars_mean
search__query_chars_median search__query_chars_std search__query_chars_min
search__query_chars_max search__query_chars_q90 search__query_tokens_mean
search__query_tokens_median search__query_tokens_std search__query_tokens_min
search__query_tokens_max search__digit_share search__alpha_share
search__space_share search__cyrillic_share search__latin_share
search__page_mean search__page_median search__page_std search__page_min
search__page_max search__page_q90 search__n_unique_pages
search__page_increment_share search__page_nondecreasing_share
search__page_reset_share search__page_repeat_share search__deep_page_share
ua__family_app_share ua__family_headless_share ua__family_yandex_share
ua__family_firefox_share ua__family_chrome_share ua__family_safari_share
ua__family_http_client_share ua__family_other_share ua__os_android_share
ua__os_ios_share ua__os_windows_share ua__os_mac_share ua__os_linux_share
ua__os_other_share ua__mobile_share ua__headless_share ua__http_client_share
ua__bot_keyword_share ua__tablet_share ua__platform_web_share
ua__platform_android_share ua__platform_ios_share ua__platform_unknown_share
ua__n_unique_raw_ua ua__n_unique_platform ua__ua_entropy
ua__ua_normalized_entropy ua__ua_hhi ua__ua_top1_share ua__ua_top3_share
ua__ua_change_share ua__platform_os_mismatch_share pointer__n_points
pointer__coverage_all pointer__coverage_web pointer__x_mean pointer__x_median
pointer__x_std pointer__x_min pointer__x_max pointer__x_q10 pointer__x_q25
pointer__x_q75 pointer__x_q90 pointer__x_nunique pointer__y_mean
pointer__y_median pointer__y_std pointer__y_min pointer__y_max pointer__y_q10
pointer__y_q25 pointer__y_q75 pointer__y_q90 pointer__y_nunique
pointer__n_unique_pairs pointer__repeat_pair_share
pointer__dominant_pair_share pointer__bbox_width pointer__bbox_height
pointer__bbox_area pointer__distance_mean pointer__distance_median
pointer__distance_std pointer__distance_max pointer__path_sum
pointer__abs_dx_mean pointer__abs_dy_mean pointer__zero_movement_share
pointer__same_x_share pointer__same_y_share pointer__path_efficiency
ratios__unique_items_per_event ratios__views_per_search
ratios__contacts_per_view ratios__favorites_per_view ratios__photos_per_view
ratios__seller_views_per_view ratios__queries_per_active_minute
""".split()

HISTORY_FEATURES = """
item_history__seen_before_share item_history__prior_count_mean
item_history__prior_count_max item_history__prior_frequency_mean
item_history__seen_by_3_share item_history__recent_7d_count_mean
item_history__recent_7d_seen_share item_history__max_cookie_overlap
item_history__max_cookie_overlap_share item_history__max_cookie_jaccard
item_history__prior_cookie_overlap_3_count
""".split()

SELECTED_FEATURES = """
basic__n_events basic__n_event_types metadata__age_days
event__contact_chat_open_share event__contact_message_sent_count
event__contact_message_sent_share event__contact_phone_show_count
event__contact_phone_show_share event__contact_phone_show_present
event__contact_phone_show_log_count event__favorite_add_count
event__favorite_add_share event__favorite_add_present
event__favorite_add_log_count event__item_view_count event__item_view_share
event__login_count event__login_share event__photo_swipe_share
event__search_results_view_count event__search_results_view_share
event__search_results_view_present event__seller_page_view_count
event__seller_page_view_share entropy__event_entropy
entropy__event_normalized_entropy entropy__event_hhi entropy__event_top1_share
entropy__event_top3_share temporal__active_duration temporal__first_offset
temporal__last_offset temporal__tail_gap temporal__events_per_active_second
temporal__events_per_window_hour temporal__delta_mean temporal__delta_median
temporal__delta_std temporal__delta_min temporal__delta_max
temporal__delta_q01 temporal__delta_q05 temporal__delta_q10
temporal__delta_q25 temporal__delta_q50 temporal__delta_q75
temporal__delta_q90 temporal__delta_q95 temporal__delta_q99 temporal__delta_cv
temporal__burstiness temporal__n_unique_delta temporal__delta_lt_2_share
temporal__delta_lt_5_share temporal__delta_lt_10_share
temporal__delta_lt_30_share temporal__delta_mode_share
temporal__delta_repeat_share activity__bucket_1s_q95
entropy__bucket_1s_entropy activity__bucket_1s_top1_share
activity__bucket_60s_mean activity__bucket_60s_std activity__bucket_60s_median
activity__bucket_60s_q95 entropy__bucket_60s_entropy
activity__bucket_60s_top1_share activity__bucket_300s_active
activity__bucket_300s_mean activity__bucket_300s_std
activity__bucket_300s_median activity__bucket_300s_q95
entropy__bucket_300s_entropy activity__bucket_300s_top1_share
activity__bucket_900s_mean activity__bucket_900s_std
activity__bucket_900s_median activity__bucket_900s_q95
entropy__bucket_900s_entropy activity__bucket_900s_top1_share
activity__bucket_3600s_mean activity__bucket_3600s_std
activity__bucket_3600s_max activity__bucket_3600s_median
activity__bucket_3600s_q95 entropy__bucket_3600s_entropy
activity__bucket_3600s_top1_share activity__session_30s_count
activity__session_30s_mean_duration activity__session_30s_max_duration
activity__session_30s_duration_std activity__session_30s_mean_events
activity__session_60s_count activity__session_60s_mean_duration
activity__session_60s_max_duration activity__session_60s_duration_std
activity__session_60s_mean_events activity__session_60s_max_events
activity__session_300s_count activity__session_300s_mean_duration
activity__session_300s_max_duration activity__session_300s_duration_std
activity__session_300s_mean_events activity__session_300s_max_events
activity__session_1800s_count activity__session_1800s_mean_duration
activity__session_1800s_max_duration activity__session_1800s_duration_std
activity__session_1800s_mean_events activity__session_1800s_max_events
sequence__n_transitions sequence__n_unique_transitions
sequence__repeat_transition_share sequence__self_transition_share
sequence__transition_entropy sequence__transition_normalized_entropy
sequence__transition_hhi sequence__transition_top1_share
sequence__transition_top3_share sequence__longest_run sequence__mean_run
sequence__n_runs sequence__bigram_0_share sequence__bigram_1_share
sequence__bigram_2_share sequence__bigram_3_share sequence__bigram_4_share
sequence__bigram_5_share sequence__bigram_6_share sequence__bigram_7_share
sequence__bigram_8_share sequence__bigram_9_share sequence__bigram_10_share
sequence__bigram_11_share item__events_per_item item__revisit_count
item__item_n_unique item__item_missing_share item__item_entropy
item__item_normalized_entropy item__item_hhi item__item_top1_share
item__item_top3_share item__category_n_unique item__category_missing_share
item__category_entropy item__category_normalized_entropy item__category_hhi
item__category_top1_share item__category_top3_share item__location_n_unique
item__location_missing_share item__location_entropy
item__location_normalized_entropy item__location_hhi item__location_top1_share
item__location_top3_share item__seller_missing_share item__seller_entropy
item__seller_hhi item__seller_pro_share item__consecutive_same_item
item__consecutive_same_category item__consecutive_same_location
search__n_unique_queries search__repeat_query_share search__query_entropy
search__query_normalized_entropy search__query_hhi search__query_top1_share
search__query_top3_share search__query_chars_mean search__query_chars_median
search__query_chars_std search__query_chars_min search__query_chars_max
search__query_chars_q90 search__query_tokens_mean search__query_tokens_std
search__query_tokens_max search__digit_share search__alpha_share
search__space_share search__cyrillic_share search__latin_share
search__page_mean search__page_median search__page_std search__page_min
search__page_max search__page_q90 search__n_unique_pages
search__page_increment_share search__page_nondecreasing_share
search__page_reset_share search__deep_page_share ua__family_app_share
ua__family_yandex_share ua__family_chrome_share ua__os_ios_share
ua__os_windows_share ua__os_mac_share pointer__n_points pointer__coverage_web
pointer__x_mean pointer__x_median pointer__x_std pointer__x_min pointer__x_max
pointer__x_q10 pointer__x_q25 pointer__x_q75 pointer__x_q90 pointer__x_nunique
pointer__y_mean pointer__y_median pointer__y_std pointer__y_min pointer__y_max
pointer__y_q10 pointer__y_q25 pointer__y_q90 pointer__y_nunique
pointer__bbox_width pointer__bbox_area pointer__distance_mean
pointer__distance_median pointer__distance_std pointer__distance_max
pointer__path_sum pointer__abs_dx_mean pointer__abs_dy_mean
pointer__path_efficiency ratios__unique_items_per_event
ratios__views_per_search ratios__contacts_per_view ratios__favorites_per_view
ratios__photos_per_view ratios__seller_views_per_view
ratios__queries_per_active_minute
""".split()

warmup = train.loc[train.window_start_ts < "2026-04-10"]
feature_config = fit_config(events, warmup)
behavior = build_features(events, metadata, feature_config)
history = build_history_features(events, metadata)
matrix = pd.concat([behavior, history], axis=1)

full_columns = BASE_FEATURES + HISTORY_FEATURES
selected_columns = SELECTED_FEATURES + HISTORY_FEATURES
train_matrix = matrix.loc[train.cookie_id]
print("feature matrix:", matrix.shape)
print("model columns:", len(full_columns), len(selected_columns))


feature matrix: (16000, 346)
model columns: 331 251


## Временная валидация и модель

В каждом фолде обучающие окна закончились до начала проверочных.
Категориальные частоты и медианы вычисляются на обучающей части фолда.
Две модели `HistGradientBoostingClassifier` используют полный и
сокращённый списки признаков; их прогнозы усредняются 50/50.

In [7]:
"""Precision при заданном Recall по куки с учётом групп одинаковых score.

Расчёт написан по формуле задачи и сверяется с выданной реализацией метрики.
Здесь score может быть любым конечным числом.
Диапазон значений в файле ответа проверяется отдельно.
"""

from __future__ import annotations

import numpy as np


def precision_at_recall(
    y_true, y_score, min_recall=0.70, *, return_curve=False
):
    """Найти максимальный precision и соответствующий порог.

    Одинаковые score включаются одной группой. При равном precision берётся
    наибольший порог. Если положительных меток нет, результат — NaN,
    как в официальной метрике; порог не выбирается. Некорректные входы
    вызывают ошибку, а не приводятся к целым числам.
    """
    y = np.asarray(y_true)
    s = np.asarray(y_score, dtype=float)
    r = float(min_recall)
    if y.ndim != 1 or s.ndim != 1 or y.shape != s.shape:
        raise ValueError("y_true and y_score must be equal-length 1D arrays")
    if not np.isin(y, [0, 1]).all():
        raise ValueError("y_true must contain only 0 and 1")
    if not np.isfinite(s).all():
        raise ValueError("y_score must be finite")
    if not np.isfinite(r) or not 0 <= r <= 1:
        raise ValueError("min_recall must be finite and in [0, 1]")
    y = y.astype(np.int64, copy=False)
    n_pos = int(y.sum())
    if n_pos == 0:
        result = {
            "score": float("nan"),
            "threshold": None,
            "precision": float("nan"),
            "recall": float("nan"),
            "tp": None,
            "fp": None,
            "fn": None,
            "predicted_positive_count": None,
            "status": "undefined_no_positives",
        }
        if return_curve:
            result["curve"] = {
                k: []
                for k in (
                    "threshold",
                    "precision",
                    "recall",
                    "tp",
                    "fp",
                    "fn",
                    "predicted_positive_count",
                )
            }
        return result

    order = np.argsort(-s, kind="mergesort")
    sorted_scores = s[order]
    group_ends = np.flatnonzero(
        np.r_[sorted_scores[1:] != sorted_scores[:-1], True]
    )
    tp = np.cumsum(y[order], dtype=np.int64)[group_ends]
    counts = group_ends + 1
    precision = tp / counts
    recall = tp / n_pos
    feasible = np.flatnonzero(recall >= r)
    best = int(feasible[np.argmax(precision[feasible])])
    result = {
        "score": float(precision[best]),
        "threshold": float(sorted_scores[group_ends[best]]),
        "precision": float(precision[best]),
        "recall": float(recall[best]),
        "tp": int(tp[best]),
        "fp": int(counts[best] - tp[best]),
        "fn": int(n_pos - tp[best]),
        "predicted_positive_count": int(counts[best]),
        "status": "ok",
    }
    if return_curve:
        result["curve"] = {
            "threshold": sorted_scores[group_ends].tolist(),
            "precision": precision.tolist(),
            "recall": recall.tolist(),
            "tp": tp.tolist(),
            "fp": (counts - tp).tolist(),
            "fn": (n_pos - tp).tolist(),
            "predicted_positive_count": counts.tolist(),
        }
    return result


In [8]:
"""Выбранная модель HistGradientBoosting и проверка на следующих периодах."""

from __future__ import annotations

import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline


class FoldFrequencyEncoder(BaseEstimator, TransformerMixin):
    """Считать частоты категорий только по данным обучения текущего фолда."""

    def fit(self, X, y=None):
        self.columns_ = list(X.columns)
        self.maps_ = {
            c: X[c].astype(str).value_counts().to_dict()
            for c in X
            if c.startswith("rarity__")
        }
        return self

    def transform(self, X):
        result = X[self.columns_].copy()
        for column, mapping in self.maps_.items():
            result[column] = np.log1p(
                result[column].astype(str).map(mapping).fillna(0)
            )
        return result.replace([np.inf, -np.inf], np.nan).astype("float64")


def make_model(seed: int):
    return Pipeline(
        [
            ("frequency", FoldFrequencyEncoder()),
            (
                "imputer",
                SimpleImputer(strategy="median", keep_empty_features=True),
            ),
            (
                "model",
                HistGradientBoostingClassifier(
                    max_iter=440,
                    learning_rate=0.04,
                    max_leaf_nodes=15,
                    min_samples_leaf=25,
                    l2_regularization=3.0,
                    early_stopping=False,
                    random_state=seed,
                ),
            ),
        ]
    )


class MeanBlend:
    """Среднее прогнозов двух моделей; у каждой свой список признаков."""

    def __init__(self, baseline, selected, baseline_columns, selected_columns):
        self.baseline = baseline
        self.selected = selected
        self.baseline_columns = list(baseline_columns)
        self.selected_columns = list(selected_columns)

    def predict_proba(self, matrix):
        first = self.baseline.predict_proba(matrix[self.baseline_columns])[
            :, 1
        ]
        second = self.selected.predict_proba(matrix[self.selected_columns])[
            :, 1
        ]
        score = (first + second) / 2
        return np.column_stack([1 - score, score])


def temporal_splits(metadata, secondary: bool = False):
    bounds = (
        [("2026-04-17", "2026-04-20")]
        if secondary
        else [
            ("2026-04-10", "2026-04-12"),
            ("2026-04-12", "2026-04-14"),
            ("2026-04-14", "2026-04-17"),
        ]
    )
    for fold, (start, end) in enumerate(bounds):
        fit = np.flatnonzero(metadata.window_end_ts.le(start))
        valid = np.flatnonzero(
            metadata.window_start_ts.ge(start)
            & metadata.window_start_ts.lt(end)
        )
        assert len(fit) and len(valid)
        assert (
            metadata.iloc[fit].window_end_ts.max()
            <= metadata.iloc[valid].window_start_ts.min()
        )
        yield fold, fit, valid


def score_summary(y, scores):
    return {
        "P_R70": precision_at_recall(y, scores),
        "PR_AUC": float(average_precision_score(y, scores)),
        "ROC_AUC": float(roc_auc_score(y, scores)),
    }


In [9]:
def out_of_fold(columns):
    prediction = np.full(len(train), np.nan)
    fold_ids = np.full(len(train), -1)
    with threadpool_limits(limits=2):
        for late in (False, True):
            for fold, fit, valid in temporal_splits(train, secondary=late):
                fitted = make_model(MODEL_SEED)
                fitted.fit(
                    train_matrix.iloc[fit][columns], train.target.iloc[fit]
                )
                prediction[valid] = fitted.predict_proba(
                    train_matrix.iloc[valid][columns]
                )[:, 1]
                fold_ids[valid] = fold + (3 if late else 0)
    covered = np.isfinite(prediction)
    return pd.DataFrame(
        {
            "cookie_id": train.cookie_id[covered].to_numpy(),
            "target": train.target[covered].to_numpy(),
            "fold": fold_ids[covered],
            "score": prediction[covered],
        }
    )


full_oof = out_of_fold(full_columns)
selected_oof = out_of_fold(selected_columns)
oof = full_oof.copy()
oof["score"] = (full_oof.score + selected_oof.score) / 2
for name, part in (
    ("primary", oof[oof.fold < 3]),
    ("secondary", oof[oof.fold == 3]),
):
    part.to_csv(output / f"oof_{name}.csv", index=False, float_format="%.12g")

with threadpool_limits(limits=2):
    full_model = make_model(MODEL_SEED)
    selected_model = make_model(MODEL_SEED)
    full_model.fit(train_matrix[full_columns], train.target)
    selected_model.fit(train_matrix[selected_columns], train.target)
    test_matrix = matrix.loc[test.cookie_id]
    test_score = (
        full_model.predict_proba(test_matrix[full_columns])[:, 1]
        + selected_model.predict_proba(test_matrix[selected_columns])[:, 1]
    ) / 2

sample = pd.read_csv(root / "data/sample_submission.csv")
base_submission = sample[["cookie_id"]].copy()
base_submission["score"] = base_submission.cookie_id.map(
    pd.Series(test_score, index=test.cookie_id)
)
base_submission.to_csv(
    output / "submission_base.csv", index=False, float_format="%.12g"
)
print(
    "primary blend:",
    score_summary(
        oof.loc[oof.fold < 3, "target"],
        oof.loc[oof.fold < 3, "score"],
    )["P_R70"]["score"],
)


primary blend: 0.8226600985221675


## Сглаживание по общим объявлениям

Для каждого просмотренного объявления берётся средний **исходный** score
прошлых кук. При двух или более объявлениях итоговый score равен
`0.8 × base + 0.2 × neighbor`. Популярные объявления получают меньший вес.
Скорректированные прогнозы не возвращаются в историю.

In [10]:
"""Сглаживание score по общим объявлениям без меток и будущих данных."""

from __future__ import annotations

from collections import defaultdict

import numpy as np
import pandas as pd


def blend_scores(base, neighbor, *, alpha: float, space: str = "probability"):
    """Смешать прогнозы как вероятности или логарифмы шансов."""
    base = np.asarray(base, dtype=float)
    neighbor = np.asarray(neighbor, dtype=float)
    if not 0 <= alpha <= 1:
        raise ValueError("alpha must be in [0, 1]")
    if space == "probability":
        return (1 - alpha) * base + alpha * neighbor
    if space == "logit":
        eps = 1e-6
        first = np.clip(base, eps, 1 - eps)
        second = np.clip(neighbor, eps, 1 - eps)
        mixed = (1 - alpha) * np.log(first / (1 - first)) + alpha * np.log(
            second / (1 - second)
        )
        return 1 / (1 + np.exp(-mixed))
    raise ValueError("space must be 'probability' or 'logit'")


def build_viewed_item_sets(
    events: pd.DataFrame, metadata: pd.DataFrame
) -> dict[str, frozenset]:
    """Собрать уникальные объявления, просмотренные внутри окна куки."""
    ev = window_events(events, metadata, deduplicate=True)
    viewed = ev.loc[
        ev.event_name.eq("item_view") & ev.item_id.notna(),
        ["cookie_id", "item_id"],
    ].drop_duplicates()
    grouped = (
        viewed.groupby("cookie_id", sort=False)
        .item_id.agg(frozenset)
        .to_dict()
    )
    return {
        cookie: grouped.get(cookie, frozenset())
        for cookie in metadata.cookie_id
    }


def causal_item_score_smoothing(
    frame: pd.DataFrame,
    item_sets: dict[str, frozenset],
    *,
    alpha: float = 0.20,
    min_shared_items: int = 2,
) -> pd.DataFrame:
    """Смешать исходный score с прогнозами прошлых куки по общим объявлениям.

    Для текущей строки доступны только окна, завершившиеся к её началу.
    По каждому объявлению усредняются исходные score прошлых куки. Средние
    смешиваются с весами, убывающими при росте популярности объявления.
    Скорректированные score в историю не попадают.
    """
    required = {"cookie_id", "window_start_ts", "window_end_ts", "score"}
    missing = required - set(frame)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    if not 0 <= alpha <= 1:
        raise ValueError("alpha must be in [0, 1]")
    if not isinstance(min_shared_items, int) or min_shared_items < 1:
        raise ValueError("min_shared_items must be a positive integer")

    data = frame.copy().reset_index(drop=True)
    if data.cookie_id.isna().any() or not data.cookie_id.is_unique:
        raise ValueError("cookie_id must be non-null and unique")
    for column in ["window_start_ts", "window_end_ts"]:
        data[column] = pd.to_datetime(data[column], errors="raise")
    score = data.score.to_numpy(dtype=float)
    if (
        not np.isfinite(score).all()
        or not np.logical_and(score >= 0, score <= 1).all()
    ):
        raise ValueError("score must be finite and in [0, 1]")
    if not (data.window_start_ts < data.window_end_ts).all():
        raise ValueError(
            "Every observation window must have positive duration"
        )

    data["position"] = np.arange(len(data))
    by_start = data.sort_values(
        ["window_start_ts", "cookie_id"], kind="mergesort"
    )
    by_end = data.sort_values(
        ["window_end_ts", "cookie_id"], kind="mergesort"
    ).reset_index(drop=True)

    item_score_sum: dict[object, float] = defaultdict(float)
    item_count: dict[object, int] = defaultdict(int)
    item_latest_end: dict[object, pd.Timestamp] = {}
    neighbor_score = np.full(len(data), np.nan)
    shared_item_support = np.zeros(len(data), dtype=np.int64)
    max_source_end = np.full(
        len(data), np.datetime64("NaT"), dtype="datetime64[ns]"
    )
    next_completed = 0

    for start, group in by_start.groupby("window_start_ts", sort=True):
        while (
            next_completed < len(by_end)
            and by_end.iloc[next_completed].window_end_ts <= start
        ):
            old = by_end.iloc[next_completed]
            for item in item_sets.get(old.cookie_id, frozenset()):
                item_score_sum[item] += float(old.score)
                item_count[item] += 1
                item_latest_end[item] = old.window_end_ts
            next_completed += 1

        for row in group.itertuples(index=False):
            values: list[float] = []
            weights: list[float] = []
            latest: list[pd.Timestamp] = []
            for item in item_sets.get(row.cookie_id, frozenset()):
                degree = item_count.get(item, 0)
                if degree:
                    values.append(item_score_sum[item] / degree)
                    # Популярное объявление слабее указывает на сходство.
                    weights.append(1.0 / np.log1p(degree + 1))
                    latest.append(item_latest_end[item])
            position = row.position
            shared_item_support[position] = len(values)
            if values:
                neighbor_score[position] = float(
                    np.average(values, weights=weights)
                )
                max_source_end[position] = np.datetime64(max(latest))

    adjusted = shared_item_support >= min_shared_items
    graph_score = score.copy()
    graph_score[adjusted] = blend_scores(
        score[adjusted],
        neighbor_score[adjusted],
        alpha=alpha,
        space="probability",
    )
    result = data.drop(columns="position")
    result["neighbor_score"] = neighbor_score
    result["shared_item_support"] = shared_item_support
    result["max_source_window_end"] = pd.to_datetime(max_source_end)
    result["adjusted"] = adjusted
    result["score_graph"] = graph_score
    available = result.max_source_window_end.notna()
    if not (
        result.loc[available, "max_source_window_end"]
        <= result.loc[available, "window_start_ts"]
    ).all():
        raise AssertionError("Post-processing used a future observation")
    if not result.score_graph.between(0, 1).all():
        raise AssertionError("Convex score blend left [0, 1]")
    return result


In [11]:
GRAPH_ALPHA = 0.2
MIN_SHARED_ITEMS = 2
item_sets = build_viewed_item_sets(events, metadata)
oof = pd.concat(
    [
        pd.read_csv(output / f"oof_{name}.csv")
        for name in ("primary", "secondary")
    ],
    ignore_index=True,
).merge(
    train[["cookie_id", "window_start_ts", "window_end_ts"]],
    on="cookie_id",
)
oof_graph = causal_item_score_smoothing(
    oof,
    item_sets,
    alpha=GRAPH_ALPHA,
    min_shared_items=MIN_SHARED_ITEMS,
)
oof_graph.to_csv(
    output / "oof_item_graph.csv", index=False, float_format="%.12g"
)

base_submission = pd.read_csv(
    output / "submission_base.csv", float_precision="round_trip"
)
test_base = test[["cookie_id", "window_start_ts", "window_end_ts"]].merge(
    base_submission, on="cookie_id"
)
all_scores = pd.concat(
    [
        oof[["cookie_id", "window_start_ts", "window_end_ts", "score"]],
        test_base[["cookie_id", "window_start_ts", "window_end_ts", "score"]],
    ],
    ignore_index=True,
)
smoothed = causal_item_score_smoothing(
    all_scores,
    item_sets,
    alpha=GRAPH_ALPHA,
    min_shared_items=MIN_SHARED_ITEMS,
)
test_graph = smoothed.loc[smoothed.cookie_id.isin(test.cookie_id)]
submission = sample[["cookie_id"]].copy()
submission["score"] = submission.cookie_id.map(
    test_graph.set_index("cookie_id").score_graph
)
submission.to_csv(output / "submission.csv", index=False, float_format="%.12g")
print("adjusted test cookies:", int(test_graph.adjusted.sum()))


adjusted test cookies: 3169


## Результат на временных периодах

Основной результат объединяет фолды 0–2. Число по фолду 3 служит
дополнительной проверкой устойчивости: этот период участвовал в выборе
финального варианта. Метрика ниже считается заново из полученных OOF.

In [12]:
def metric_line(name, frame):
    base = precision_at_recall(frame.target, frame.score)["score"]
    final = precision_at_recall(frame.target, frame.score_graph)
    return {
        "period": name,
        "rows": len(frame),
        "base": base,
        "final": final["score"],
        "gain": final["score"] - base,
    }


rows = [
    metric_line(f"fold {fold}", part)
    for fold, part in oof_graph.groupby("fold", sort=True)
]
rows += [
    metric_line("primary 0–2", oof_graph[oof_graph.fold < 3]),
    metric_line("late 3", oof_graph[oof_graph.fold == 3]),
]
results = pd.DataFrame(rows)
print(
    results.to_string(index=False, float_format=lambda value: f"{value:.6f}")
)
primary = oof_graph[oof_graph.fold < 3]
operating_point = precision_at_recall(primary.target, primary.score_graph)
print(
    "primary TP/FP/FN:", *(operating_point[key] for key in ("tp", "fp", "fn"))
)


     period  rows     base    final     gain
     fold 0  1808 0.801418 0.812950 0.011531
     fold 1  1660 0.841584 0.848485 0.006901
     fold 2  2367 0.821429 0.842424 0.020996
     fold 3  1951 0.862595 0.875969 0.013374
primary 0–2  5835 0.822660 0.830049 0.007389
     late 3  1951 0.862595 0.875969 0.013374
primary TP/FP/FN: 337 69 139


На объединённых фолдах 0–2 сглаживание прибавило 0,007389 к целевой
метрике. Парный bootstrap по дням (5000 повторов) дал интервал прироста
[0,002495; 0,040458]. PR-AUC и ROC-AUC после сглаживания снизились,
поэтому улучшение относится именно к выбранной рабочей точке.
Интервал не учитывает выбор метода по этим же фолдам.

## Проверка ответа и границы вывода

Файл содержит одну строку на тестовую куку, две колонки и конечные score
в диапазоне [0, 1]. На основных временных фолдах получено P@R≥70% = 0,830049;
качество на скрытом тесте неизвестно. Прогнозы предназначены для
ранжирования, отдельная калибровка вероятностей не проверялась.

In [13]:
answer = pd.read_csv(output / "submission.csv")
valid = (
    list(answer.columns) == ["cookie_id", "score"]
    and answer.cookie_id.tolist() == sample.cookie_id.tolist()
    and answer.score.notna().all()
    and answer.score.between(0, 1).all()
)
if not valid:
    raise ValueError("Некорректный файл ответа")
print("submission rows:", len(answer))
print("score range:", answer.score.min(), answer.score.max())


submission rows: 4909
score range: 0.000172471220447 0.983301065349
